# Level 4 RTL and verification

Read the controller, then execute its actual tests. Requires Icarus Verilog and VVP.

In [1]:
from pathlib import Path
import sys, json, csv, subprocess
cwd=Path.cwd().resolve()
ROOT=next(p for b in [cwd,*cwd.parents] for p in [b,b/'Phase1'] if (p/'level4/rtl/ai_cache.sv').is_file())
print('Phase 1 source tree located')

Phase 1 source tree located


## Cache controller
IDLE chooses demand first. SEND holds a request through backpressure. WAIT fills a line. Selected speculation is nonpreemptible.

In [2]:
source=(ROOT/'level4/rtl/ai_cache.sv').read_text().splitlines()
print('\n'.join(f'{i+1:3} {line}' for i,line in enumerate(source) if 'assign ' in line or 'SEND:' in line or 'WAIT:' in line or 'IDLE:' in line))

 37  assign cpu_req_ready=!rst && !demand_pending && (!cpu_rsp_valid || cpu_rsp_ready)
 39  assign mem_req_valid=!rst && state==SEND;
 40  assign mem_req_addr={tx_line,4'b0};
 41  assign mem_rsp_ready=!rst && state==WAIT;
 81     IDLE: begin
 98     SEND: if(mem_req_ready) begin
102     WAIT: if(mem_rsp_valid) begin


## Run actual data and protocol checks

In [3]:
run=subprocess.run(['make','rtl-test'],cwd=ROOT,text=True,capture_output=True)
print('\n'.join(line for line in (run.stdout+run.stderr).splitlines() if 'PASS:' in line))
if run.returncode: raise RuntimeError(run.stdout+run.stderr)
print('All RTL verification commands passed')

PASS: 3673 verified responses; all directed checks passed
PASS: continuous-request stress, 2000 ordered responses, 14871 cycles
PASS: predictor training, repeated lines, phase change, signed stride, bounds and disable
PASS: next-line distinct observations, arbitrary jumps, enable, upper bound and reset
PASS: adversarial: 10000 responses, 299876 cycles, 8464 reads, hits=2029 misses=7971 PF=1049
PASS: limitation demonstration: unrelated demand accepted but held for 20 cycles behind unaccepted prefetch 0x1030 in SEND.
PASS: trace build/verification/high_latency.mem pf=0 policy=1 checked=3 completion=412 quiescent=414
PASS: trace build/verification/high_latency.mem pf=1 policy=1 checked=3 completion=412 quiescent=543
PASS: trace build/verification/same_line.mem pf=0 policy=1 checked=2 completion=139 quiescent=141
PASS: trace build/verification/same_line.mem pf=1 policy=1 checked=2 completion=139 quiescent=141
PASS: trace build/verification/high_latency.mem pf=0 policy=2 checked=3 completio

## Synthesis evidence
Generic cells describe a tool mapping, not physical area, timing closure or an FPGA board test.

In [4]:
synthesis=json.loads((ROOT/'level4/synth/reference/summary.json').read_text())
for row in synthesis['runs']:print(row['policy'],row['generic_cells'],'generic cells;',row['check_problems'],'CHECK problems')

stride 33245 generic cells; 0 CHECK problems
next_line 32877 generic cells; 0 CHECK problems
